# requires-grad-propagation — ex1: three-gate requires_grad: toggle AND is_differentiable AND any-input

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `requires-grad-propagation`. Running the final beacon cell reports progress against the `Backprop: requires_grad propagation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: requires_grad propagation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`requires-grad-propagation`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "requires-grad-propagation"
DD_SUBTOPIC = "Backprop: requires_grad propagation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## requires_grad propagation — quick refresher

Output `requires_grad` is the **OR over all Tensor inputs** — if ANY input is grad-tracked, the output must be too (otherwise we lose the chain):

```python
requires_grad = grad_tracking_enabled and is_differentiable and any(
    isinstance(a, Tensor) and a.requires_grad for a in args
)
```

Three gates, ALL must be true:
1. `grad_tracking_enabled` (the global toggle).
2. `is_differentiable` (the per-op flag; ops like `torch.equal`    pass `False`).
3. At least one Tensor input with `requires_grad=True`.

Non-Tensor inputs (ints, floats) are filtered by the `isinstance` guard so they don't accidentally veto grad. Constants don't *contribute* grad tracking — but they don't *block* it either.

### Exercise 1 — three-gate requires_grad: toggle AND is_differentiable AND any-input

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the three-gate requires_grad rule (global toggle AND op differentiability AND any-input-requires-grad) and filter non-Tensor inputs out of the OR-reduction.
> Keywords: requires-grad, propagation, is_differentiable, any, three-gate
> ```

**KCs targeted:** `requires-grad-propagation`, `grad-tracking-global-toggle`

Implement `propagate_requires_grad(args, is_differentiable, grad_tracking_enabled)`. Output `requires_grad` is the AND of THREE gates — ALL must be true:

1. `grad_tracking_enabled` — the global no-grad toggle.
2. `is_differentiable` — the per-op flag (e.g. `t.equal` registers with `is_differentiable=False`).
3. **At least one input is a Tensor with `requires_grad=True`.**
   Use `any(isinstance(a, Tensor) and a.requires_grad for a in args)`. The `isinstance` guard is critical — without it you'd ask non-Tensors (ints, floats, shape tuples) for `.requires_grad` and crash with `AttributeError`.

Signature: `propagate_requires_grad(args: tuple, is_differentiable: bool, grad_tracking_enabled: bool) -> bool`.

Inputs vary in type: some are `Tensor`, some are Python scalars or tuples. Constants must NOT veto grad — `multiply(t, 3.0)` with `t .requires_grad=True` should produce a grad-tracked output. They just don't *contribute* a True to the `any`.

Test cases exhaustively cover the truth table:
- All gates True with at least one tracked Tensor → True.
- Any gate False → False (3 scenarios).
- Mixed Tensor/non-Tensor where the non-Tensor would crash a naive implementation.

In [ ]:
def propagate_requires_grad(
    args: tuple,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    return (
        grad_tracking_enabled
        and is_differentiable
        and any(
            isinstance(a, Tensor) and a.requires_grad for a in args
        )
    )


<details><summary>Solution</summary>

```python
def propagate_requires_grad(
    args: tuple,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    return (
        grad_tracking_enabled
        and is_differentiable
        and any(
            isinstance(a, Tensor) and a.requires_grad for a in args
        )
    )
```

**Why AND, not OR.** Each gate is a *necessary* condition. The global toggle has to be on (otherwise we're in no_grad). The op has to be differentiable (no point recording a Recipe for `torch.equal` — gradients don't flow through booleans). And at least one input has to require grad (otherwise the output is constant w.r.t. all params — no graph needed).

**Why `isinstance(a, Tensor) AND a.requires_grad` inside `any`.** Short-circuit evaluation: `isinstance` is cheap and false for non-Tensors → `and` skips the `.requires_grad` access. Without the guard, `propagate_requires_grad((my_tensor, 3.0), ...)` crashes with `AttributeError: 'float' object has no attribute 'requires_grad'`.

**`is_differentiable` lives on the op, not the inputs.** `wrap_forward_fn(torch.equal, is_differentiable=False)` registers `torch.equal` with the flag set to False on the wrapper. The flag is plumbed into `propagate_requires_grad` from there, not from any inspection of the inputs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()